In [4]:
import torch
from transformers import GPT2LMHeadModel

hf_model=GPT2LMHeadModel.from_pretrained("gpt2")
hf_sd=hf_model.state_dict()

converted_sd={}

transposed=(
    "attn.c_attn.weight",
    "attn.c_proj.weight",
    "mlp.c_fc.weight",
    "mlp.c_proj.weight",
)

for k,v in hf_sd.items():
    if any(k.endswith(w) for w in transposed):
        converted_sd[k]=v.t().contiguous()
    else:
        converted_sd[k]=v

torch.save(converted_sd,"gpt2_openai_weights.pt")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [1]:
import torch

# Input and output checkpoints
INPUT_CKPT = "gpt2_instruction_tuned.pt"
OUTPUT_CKPT = "gpt2_instruction_tuned_transposed.pt"

# Load checkpoint
ckpt = torch.load(INPUT_CKPT, map_location="cpu")

# Some checkpoints are just state_dicts, others wrap them
if "model" in ckpt:
    state_dict = ckpt["model"]
elif "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# GPT-2 Conv1D weights that need transposing
TRANSPOSE_SUFFIXES = (
    "attn.c_attn.weight",
    "attn.c_proj.weight",
    "mlp.c_fc.weight",
    "mlp.c_proj.weight",
)

count = 0

for key in list(state_dict.keys()):
    if key.endswith(TRANSPOSE_SUFFIXES):
        print(
            f"Transposing {key}: "
            f"{tuple(state_dict[key].shape)} -> "
            f"{tuple(state_dict[key].t().shape)}"
        )
        state_dict[key] = state_dict[key].t().contiguous()
        count += 1

print(f"\nTransposed {count} tensors.")

# Put state_dict back if necessary
if "model" in ckpt:
    ckpt["model"] = state_dict
elif "state_dict" in ckpt:
    ckpt["state_dict"] = state_dict
else:
    ckpt = state_dict

torch.save(ckpt, OUTPUT_CKPT)
print(f"Saved to {OUTPUT_CKPT}")

Transposing transformer.h.0.attn.c_attn.weight: (768, 2304) -> (2304, 768)
Transposing transformer.h.0.attn.c_proj.weight: (768, 768) -> (768, 768)
Transposing transformer.h.0.mlp.c_fc.weight: (768, 3072) -> (3072, 768)
Transposing transformer.h.0.mlp.c_proj.weight: (3072, 768) -> (768, 3072)
Transposing transformer.h.1.attn.c_attn.weight: (768, 2304) -> (2304, 768)
Transposing transformer.h.1.attn.c_proj.weight: (768, 768) -> (768, 768)
Transposing transformer.h.1.mlp.c_fc.weight: (768, 3072) -> (3072, 768)
Transposing transformer.h.1.mlp.c_proj.weight: (3072, 768) -> (768, 3072)
Transposing transformer.h.2.attn.c_attn.weight: (768, 2304) -> (2304, 768)
Transposing transformer.h.2.attn.c_proj.weight: (768, 768) -> (768, 768)
Transposing transformer.h.2.mlp.c_fc.weight: (768, 3072) -> (3072, 768)
Transposing transformer.h.2.mlp.c_proj.weight: (3072, 768) -> (768, 3072)
Transposing transformer.h.3.attn.c_attn.weight: (768, 2304) -> (2304, 768)
Transposing transformer.h.3.attn.c_proj.wei